In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
from pathlib import Path

img_path = Path(r"c:\Users\v\Desktop\ECGPerturb-main\data\output_augmentation\images\ECG_033_158_p0_aug.webp")

img_native = Image.open(img_path).convert("RGB")
W, H = img_native.size

img_1024 = img_native.resize((1024, 1024), Image.BILINEAR)

fig, axes = plt.subplots(1, 2, figsize=(20, 10))

axes[0].imshow(img_native)
axes[0].set_title(f"Native : {W}x{H} px", fontsize=14)
axes[0].axis("off")

axes[1].imshow(img_1024)
axes[1].set_title("Vue modele : 1024x1024 px (resize bilineaire)", fontsize=14)
axes[1].axis("off")

plt.suptitle(img_path.name, fontsize=12)
plt.tight_layout()
plt.show()

print(f"Native     : {W} x {H} px  ({W*H/1e6:.1f} Mpx)")
print(f"Modele     : 1024 x 1024 px ({1024*1024/1e6:.1f} Mpx)")
print(f"Facteur reduction : {W/1024:.2f}x en largeur, {H/1024:.2f}x en hauteur")
print(f"Perte pixels : {(1 - 1024*1024/(W*H))*100:.1f}%")

In [ ]:
import os
os.environ["TORCH_CUDA_ARCH_LIST"] = "9.0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"


In [ ]:
import os, sys, time, json, datetime, argparse
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
torch.backends.cudnn.benchmark = False

from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
from PIL import Image

import segmentation_models_pytorch as smp
import albumentations as A

# Acces au schema NPZ partage du projet
PROJECT_ROOT = r"C:\Users\v\Desktop\ECGPerturb-main"
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
from shared.npz_schema import load_unified_npz


In [ ]:
from dataclasses import dataclass, field

@dataclass
class TrainConfig:
    # Configuration d'entrainement pour la detection de points d'intersection (5mm) via NPZ.

    # Chemins
    project_root: str = r"C:\Users\v\Desktop\ECGPerturb-main\data"
    image_dir: str = ""
    npz_dir:   str = ""
    output_dir: str = ""

    # Cible : intersections de la grille 5mm (cle NPZ)
    npz_key: str = "grid_major_5mm"
    point_radius: int = 3   # rayon (px) des disques dessines pour les ground truth

    # Architecture
    encoder_name: str = "resnet34"
    encoder_weights: str = "imagenet"
    in_channels: int = 3
    num_classes: int = 1

    # Resolution
    img_height: int = 1024
    img_width:  int = 1024

    # Entrainement
    batch_size: int = 4
    num_epochs: int = 30
    learning_rate: float = 1e-4
    weight_decay:  float = 1e-5
    num_workers: int = 0
    pin_memory:  bool = True

    # Loss & scheduler
    loss_type: str = "bce_dice"
    bce_weight: float = 0.5
    scheduler_patience: int = 5
    scheduler_factor: float = 0.5
    early_stop_patience: int = 15

    # Split par prefixe de nom de fichier
    train_sources: list = field(default_factory=lambda: ["ECG_031", "ECG_032"])
    val_sources:   list = field(default_factory=lambda: ["ECG_033"])

    device: str = ""
    seed: int = 42
    save_every_n_epochs:    int = 10
    log_images_every_n_epochs: int = 5

    def __post_init__(self):
        if not self.image_dir:
            self.image_dir = os.path.join(self.project_root, "output_augmentation", "images")
        if not self.npz_dir:
            self.npz_dir = os.path.join(self.project_root, "output_augmentation", "npz")
        if not self.output_dir:
            self.output_dir = os.path.join(self.project_root, "training", "runs_npz")
        if not self.device:
            self.device = "cuda" if torch.cuda.is_available() else "cpu"

        if self.device == "cuda":
            print(f"[OK] GPU detecte : {torch.cuda.get_device_name(0)}")
            vram = torch.cuda.get_device_properties(0).total_memory / 1e9
            print(f"   VRAM: {vram:.1f} GB")
        else:
            print("[!] Mode CPU actif - performances limitees a 1024x1024.")

cfg = TrainConfig()


In [ ]:
class ECGNpzDataset(Dataset):
    # Dataset qui apparie chaque image augmentee P2 avec un mask binaire
    # genere a la volee depuis le NPZ (points d'intersection grille major).
    # Le mask est dessine en pleine resolution image (W,H native du NPZ),
    # puis redimensionne (NEAREST) a (img_width, img_height).

    def __init__(self, image_dir, npz_dir, npz_key="grid_major_5mm",
                 source_prefixes=None, img_height=1024, img_width=1024,
                 point_radius=3, augment=False):
        self.image_dir = image_dir
        self.npz_dir   = npz_dir
        self.npz_key   = npz_key
        self.img_height, self.img_width = img_height, img_width
        self.point_radius = point_radius

        self.samples = []
        for fname in sorted(os.listdir(image_dir)):
            if not fname.endswith(".webp"):
                continue
            if source_prefixes and not any(fname.startswith(p) for p in source_prefixes):
                continue
            stem = os.path.splitext(fname)[0]
            npz_path = os.path.join(npz_dir, stem + ".npz")
            if os.path.exists(npz_path):
                self.samples.append({
                    "image_path": os.path.join(image_dir, fname),
                    "npz_path":   npz_path,
                    "stem":       stem,
                })

        # Augmentations couleur uniquement (la geometrie casserait l'alignement points/image)
        if augment:
            self.transform = A.Compose([
                A.HorizontalFlip(p=0.5),
                A.VerticalFlip(p=0.3),
                A.ColorJitter(brightness=0.15, contrast=0.15,
                              saturation=0.1, hue=0.05, p=0.5),
                A.GaussNoise(p=0.2),
            ])
        else:
            self.transform = None

        print(f"  -> {len(self.samples)} paires trouvees (key: {npz_key},"
              f" sources: {source_prefixes or 'toutes'})")

    def __len__(self):
        return len(self.samples)

    @staticmethod
    def _draw_points_mask(pts_xy, H, W, radius):
        # Dessine des disques blancs aux coordonnees pts_xy (Nx2) sur un fond noir HxW.
        mask = np.zeros((H, W), dtype=np.uint8)
        if len(pts_xy) == 0:
            return mask
        xs = np.round(pts_xy[:, 0]).astype(int)
        ys = np.round(pts_xy[:, 1]).astype(int)
        valid = (xs >= 0) & (xs < W) & (ys >= 0) & (ys < H)
        xs, ys = xs[valid], ys[valid]
        for dx in range(-radius, radius + 1):
            for dy in range(-radius, radius + 1):
                if dx*dx + dy*dy <= radius*radius:
                    mask[np.clip(ys + dy, 0, H - 1),
                         np.clip(xs + dx, 0, W - 1)] = 255
        return mask

    def __getitem__(self, idx):
        s = self.samples[idx]

        # Image
        img = Image.open(s["image_path"]).convert("RGB")
        Wn, Hn = img.size  # taille native (avant resize)
        img = img.resize((self.img_width, self.img_height), Image.BILINEAR)
        img_np = np.array(img, dtype=np.float32) / 255.0

        # Mask depuis NPZ
        data = load_unified_npz(s["npz_path"])
        pts = data.get(self.npz_key, np.empty((0, 2)))
        mask_full = self._draw_points_mask(pts, Hn, Wn, self.point_radius)
        mask_pil = Image.fromarray(mask_full).resize(
            (self.img_width, self.img_height), Image.NEAREST
        )
        mask_np = np.array(mask_pil, dtype=np.float32) / 255.0

        # Augmentations couleur
        if self.transform:
            t = self.transform(image=img_np, mask=mask_np)
            img_np, mask_np = t["image"], t["mask"]

        img_tensor  = torch.from_numpy(img_np).permute(2, 0, 1).float()
        mask_tensor = torch.from_numpy(mask_np).unsqueeze(0).float()
        return img_tensor, mask_tensor


In [ ]:
# ═══════════════ Loss ═══════════════
class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__(); self.smooth = smooth
    def forward(self, pred, target):
        ps = torch.sigmoid(pred)
        inter = (ps * target).sum(dim=(2, 3))
        union = ps.sum(dim=(2, 3)) + target.sum(dim=(2, 3))
        return 1 - ((2 * inter + self.smooth) / (union + self.smooth)).mean()

class BCEDiceLoss(nn.Module):
    def __init__(self, bce_weight=0.5):
        super().__init__()
        self.bce, self.dice, self.bce_weight = nn.BCEWithLogitsLoss(), DiceLoss(), bce_weight
    def forward(self, pred, target):
        return self.bce_weight * self.bce(pred, target) + (1 - self.bce_weight) * self.dice(pred, target)

def get_loss(loss_type, bce_weight=0.5):
    return {"bce": nn.BCEWithLogitsLoss(),
            "dice": DiceLoss(),
            "bce_dice": BCEDiceLoss(bce_weight)}[loss_type]


# ═══════════════ Metrics ═══════════════
def compute_metrics(pred, target, threshold=0.5):
    with torch.no_grad():
        pb = (torch.sigmoid(pred) > threshold).float()
        inter = (pb * target).sum(dim=(2, 3))
        ps = pb.sum(dim=(2, 3)); ts = target.sum(dim=(2, 3))
        dice = (2*inter + 1e-6) / (ps + ts + 1e-6)
        iou  = (inter + 1e-6) / (ps + ts - inter + 1e-6)
        rec  = (inter + 1e-6) / (ts + 1e-6)
        prec = (inter + 1e-6) / (ps + 1e-6)
    return {"dice": dice.mean().item(), "iou": iou.mean().item(),
            "precision": prec.mean().item(), "recall": rec.mean().item()}


# ═══════════════ Visualisation ═══════════════
def save_prediction_grid(images, masks_true, masks_pred, save_path, n=4):
    n = min(n, images.shape[0])
    fig, axes = plt.subplots(n, 4, figsize=(16, 4*n))
    if n == 1: axes = axes[np.newaxis, :]
    for i in range(n):
        img = images[i].cpu().permute(1, 2, 0).numpy()
        mt  = masks_true[i, 0].cpu().numpy()
        mp  = (torch.sigmoid(masks_pred[i, 0]).cpu().numpy() > 0.5).astype(float)
        axes[i, 0].imshow(img);                  axes[i, 0].set_title("Image augmentee", fontsize=9); axes[i, 0].axis("off")
        axes[i, 1].imshow(mt, cmap="gray", vmin=0, vmax=1); axes[i, 1].set_title("Points GT (5mm)", fontsize=9); axes[i, 1].axis("off")
        axes[i, 2].imshow(mp, cmap="gray", vmin=0, vmax=1); axes[i, 2].set_title("Points predits", fontsize=9); axes[i, 2].axis("off")
        ov = img.copy()
        tp = (mp > 0.5) & (mt > 0.5); fp = (mp > 0.5) & (mt < 0.5); fn = (mp < 0.5) & (mt > 0.5)
        ov[tp] = [0, 1, 0]; ov[fp] = [1, 0, 0]; ov[fn] = [0, 0, 1]
        axes[i, 3].imshow(ov); axes[i, 3].set_title("Overlay (V=TP, R=FP, B=FN)", fontsize=9); axes[i, 3].axis("off")
    plt.tight_layout(); plt.savefig(save_path, dpi=100, bbox_inches="tight"); plt.close()


# ═══════════════ Boucles d'entrainement ═══════════════
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train(); total_loss, n = 0, 0
    tm = {"dice": 0, "iou": 0, "precision": 0, "recall": 0}
    pbar = tqdm(loader, desc="  Train", leave=False)
    for images, masks in pbar:
        images, masks = images.to(device), masks.to(device)
        pred = model(images); loss = criterion(pred, masks)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        m = compute_metrics(pred, masks)
        total_loss += loss.item()
        for k in tm: tm[k] += m[k]
        n += 1
        pbar.set_postfix(loss=f"{loss.item():.4f}", dice=f"{m['dice']:.3f}")
    return total_loss / max(n, 1), {k: v / max(n, 1) for k, v in tm.items()}

@torch.no_grad()
def validate(model, loader, criterion, device):
    model.eval(); total_loss, n = 0, 0
    tm = {"dice": 0, "iou": 0, "precision": 0, "recall": 0}
    for images, masks in tqdm(loader, desc="  Val  ", leave=False):
        images, masks = images.to(device), masks.to(device)
        pred = model(images); loss = criterion(pred, masks)
        m = compute_metrics(pred, masks)
        total_loss += loss.item()
        for k in tm: tm[k] += m[k]
        n += 1
    return total_loss / max(n, 1), {k: v / max(n, 1) for k, v in tm.items()}


def _plot_training_curves(history, save_path):
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(14, 5))
    e = range(1, len(history["train_loss"]) + 1)
    a1.plot(e, history["train_loss"], "b-", label="Train")
    a1.plot(e, history["val_loss"],   "r-", label="Val")
    a1.set_xlabel("Epoch"); a1.set_ylabel("Loss"); a1.set_title("Loss"); a1.legend(); a1.grid(True, alpha=0.3)
    a2.plot(e, history["train_dice"], "b-",  label="Train Dice")
    a2.plot(e, history["val_dice"],   "r-",  label="Val Dice")
    a2.plot(e, history["train_iou"],  "b--", alpha=0.5, label="Train IoU")
    a2.plot(e, history["val_iou"],    "r--", alpha=0.5, label="Val IoU")
    a2.set_xlabel("Epoch"); a2.set_ylabel("Score"); a2.set_title("Dice & IoU"); a2.legend(); a2.grid(True, alpha=0.3)
    plt.tight_layout(); plt.savefig(save_path, dpi=150, bbox_inches="tight"); plt.close()


def train(cfg: TrainConfig):
    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    run_dir = os.path.join(cfg.output_dir, f"run_{timestamp}")
    os.makedirs(os.path.join(run_dir, "checkpoints"), exist_ok=True)
    os.makedirs(os.path.join(run_dir, "visualizations"), exist_ok=True)

    cfg_dict = {k: (v if isinstance(v, (int, float, bool, list, type(None))) else str(v))
                for k, v in cfg.__dict__.items()}
    with open(os.path.join(run_dir, "config.json"), "w") as f:
        json.dump(cfg_dict, f, indent=2)

    print(f"\n{'='*60}\n  Entrainement U-Net -- Detection points intersections (NPZ)\n{'='*60}")
    print(f"  Device      : {cfg.device}")
    print(f"  Resolution  : {cfg.img_height}x{cfg.img_width}")
    print(f"  Cible NPZ   : {cfg.npz_key}  (point_radius={cfg.point_radius}px)")
    print(f"  Encoder     : {cfg.encoder_name}")
    print(f"  Loss        : {cfg.loss_type}")
    print(f"  Batch size  : {cfg.batch_size}")
    print(f"  Epochs      : {cfg.num_epochs}")
    print(f"  Output      : {run_dir}\n{'='*60}\n")

    torch.manual_seed(cfg.seed); np.random.seed(cfg.seed)
    device = torch.device(cfg.device)

    print("[*] Chargement des donnees...")
    print(f"  Images: {cfg.image_dir}")
    print(f"  NPZ   : {cfg.npz_dir}")

    print(f"\n  Train (sources: {cfg.train_sources}):")
    train_ds = ECGNpzDataset(cfg.image_dir, cfg.npz_dir, cfg.npz_key,
                             cfg.train_sources, cfg.img_height, cfg.img_width,
                             cfg.point_radius, augment=True)
    print(f"  Val   (sources: {cfg.val_sources}):")
    val_ds = ECGNpzDataset(cfg.image_dir, cfg.npz_dir, cfg.npz_key,
                           cfg.val_sources, cfg.img_height, cfg.img_width,
                           cfg.point_radius, augment=False)

    if len(train_ds) == 0:
        print("\n[ERREUR] Aucune donnee d'entrainement trouvee."); return

    train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True,
                              num_workers=cfg.num_workers, pin_memory=cfg.pin_memory)
    val_loader   = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False,
                              num_workers=cfg.num_workers, pin_memory=cfg.pin_memory)

    print("\n[*] Construction du modele...")
    model = smp.Unet(encoder_name=cfg.encoder_name, encoder_weights=cfg.encoder_weights,
                     in_channels=cfg.in_channels, classes=cfg.num_classes, activation=None).to(device)
    total = sum(p.numel() for p in model.parameters())
    print(f"  Parametres: {total:,}")

    criterion = get_loss(cfg.loss_type, cfg.bce_weight)
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg.learning_rate, weight_decay=cfg.weight_decay)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", patience=cfg.scheduler_patience, factor=cfg.scheduler_factor)

    history = {k: [] for k in ["train_loss", "val_loss", "train_dice", "val_dice", "train_iou", "val_iou", "lr"]}
    best_val_dice, no_improve = 0, 0

    print(f"\n>>> Debut de l'entrainement ({cfg.num_epochs} epochs)...\n")
    t_start = time.time()

    for epoch in range(1, cfg.num_epochs + 1):
        t0 = time.time()
        lr = optimizer.param_groups[0]["lr"]

        tl, tm = train_one_epoch(model, train_loader, criterion, optimizer, device)
        vl, vm = validate(model, val_loader, criterion, device)
        scheduler.step(vl)

        history["train_loss"].append(tl); history["val_loss"].append(vl)
        history["train_dice"].append(tm["dice"]); history["val_dice"].append(vm["dice"])
        history["train_iou"].append(tm["iou"]);   history["val_iou"].append(vm["iou"])
        history["lr"].append(lr)

        print(f"Epoch {epoch:3d}/{cfg.num_epochs} | "
              f"Train Loss: {tl:.4f}  Dice: {tm['dice']:.3f} | "
              f"Val Loss: {vl:.4f}  Dice: {vm['dice']:.3f}  IoU: {vm['iou']:.3f} | "
              f"LR: {lr:.1e} | {time.time()-t0:.1f}s")

        if vm["dice"] > best_val_dice:
            best_val_dice = vm["dice"]; no_improve = 0
            torch.save({"epoch": epoch, "model_state_dict": model.state_dict(),
                        "optimizer_state_dict": optimizer.state_dict(),
                        "val_dice": best_val_dice, "config": cfg_dict},
                       os.path.join(run_dir, "checkpoints", "best_model.pth"))
            print(f"  [BEST] Nouveau meilleur modele (Dice: {best_val_dice:.4f})")
        else:
            no_improve += 1

        if epoch % cfg.save_every_n_epochs == 0:
            torch.save({"epoch": epoch, "model_state_dict": model.state_dict(),
                        "optimizer_state_dict": optimizer.state_dict(),
                        "val_dice": vm["dice"]},
                       os.path.join(run_dir, "checkpoints", f"checkpoint_epoch{epoch:03d}.pth"))

        if epoch % cfg.log_images_every_n_epochs == 0 or epoch == 1:
            model.eval()
            with torch.no_grad():
                si, sm = next(iter(val_loader))
                sp = model(si.to(device)).cpu()
                save_prediction_grid(si, sm, sp,
                    os.path.join(run_dir, "visualizations", f"epoch_{epoch:03d}.png"))

        if no_improve >= cfg.early_stop_patience:
            print(f"\n[STOP] Early stopping apres {cfg.early_stop_patience} epochs sans amelioration"); break

    total_time = time.time() - t_start
    print(f"\n{'='*60}\n  Entrainement termine en {total_time/60:.1f} minutes")
    print(f"  Meilleur Val Dice: {best_val_dice:.4f}")
    print(f"  Resultats dans: {run_dir}\n{'='*60}")
    with open(os.path.join(run_dir, "history.json"), "w") as f:
        json.dump(history, f, indent=2)
    _plot_training_curves(history, os.path.join(run_dir, "training_curves.png"))
    return run_dir


In [ ]:
# Lancement de l'entrainement
train(cfg)


In [ ]:
import glob
from IPython.display import display

runs_dir = cfg.output_dir
run_dirs = sorted(glob.glob(os.path.join(runs_dir, "run_*")))
if not run_dirs:
    raise FileNotFoundError(f"Aucun run trouve dans {runs_dir}")
run_dir = run_dirs[-1]
print(f"Run selectionne : {run_dir}")

best_path = os.path.join(run_dir, "checkpoints", "best_model.pth")
ckpt_list = sorted(glob.glob(os.path.join(run_dir, "checkpoints", "*.pth")))
checkpoint_path = best_path if os.path.exists(best_path) else (ckpt_list[-1] if ckpt_list else None)
if checkpoint_path is None:
    raise FileNotFoundError("Aucun checkpoint trouve")
print(f"Checkpoint : {checkpoint_path}")

DEVICE = cfg.device
model = smp.Unet(encoder_name=cfg.encoder_name, encoder_weights=None,
                 in_channels=cfg.in_channels, classes=cfg.num_classes)
checkpoint = torch.load(checkpoint_path, map_location=DEVICE, weights_only=False)
model.load_state_dict(checkpoint["model_state_dict"])
model = model.to(DEVICE).eval()
print(f"Modele charge - epoch {checkpoint['epoch']} | Val Dice : {checkpoint.get('val_dice', 'N/A')}")


In [ ]:
image_dir = cfg.image_dir
val_images   = sorted([f for f in os.listdir(image_dir)
                       if f.startswith("ECG_033") and f.endswith(".webp")])
train_images = sorted([f for f in os.listdir(image_dir)
                       if (f.startswith("ECG_031") or f.startswith("ECG_032"))
                       and f.endswith(".webp")])
print(f"Train: {len(train_images)} | Val: {len(val_images)}")


In [ ]:
%matplotlib inline
SET   = "val"   # "val" ou "train"
INDEX = 498

image_list = val_images if SET == "val" else train_images
if INDEX >= len(image_list):
    raise IndexError(f"INDEX={INDEX} hors limites - {len(image_list)} images dans '{SET}'")

sample_file = image_list[INDEX]
sample_path = os.path.join(image_dir, sample_file)
npz_path    = os.path.join(cfg.npz_dir, sample_file.replace(".webp", ".npz"))

print(f"Image : {sample_file}")
print(f"NPZ   : {npz_path}")
if not os.path.exists(npz_path):
    raise FileNotFoundError(f"NPZ introuvable : {npz_path}")

# === Image ===
img_pil = Image.open(sample_path).convert("RGB")
Wn, Hn = img_pil.size
img_pil = img_pil.resize((cfg.img_width, cfg.img_height), Image.BILINEAR)
img_np  = np.array(img_pil, dtype=np.float32) / 255.0

mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
std  = np.array([0.229, 0.224, 0.225], dtype=np.float32)
img_normalized = (img_np - mean) / std

# === Mask GT depuis NPZ (meme logique que ECGNpzDataset._draw_points_mask) ===
data = load_unified_npz(npz_path)
pts  = data.get(cfg.npz_key, np.empty((0, 2)))
mask_full = ECGNpzDataset._draw_points_mask(pts, Hn, Wn, cfg.point_radius)
mask_pil  = Image.fromarray(mask_full).resize((cfg.img_width, cfg.img_height), Image.NEAREST)
mask_np   = np.array(mask_pil, dtype=np.float32) / 255.0

# === Inference ===
img_tensor = torch.from_numpy(img_normalized).permute(2, 0, 1).unsqueeze(0).float().to(DEVICE)
with torch.no_grad():
    output = model(img_tensor)
    prediction = torch.sigmoid(output)
pred_np     = prediction.squeeze().cpu().numpy()
pred_binary = (pred_np > 0.5).astype(np.float32)

inter = (pred_binary * mask_np).sum()
dice  = (2 * inter) / (pred_binary.sum() + mask_np.sum() + 1e-8)
print(f"\nMax: {pred_np.max():.4f} | Moyenne: {pred_np.mean():.4f}")
print(f"Dice sur cette image : {dice:.4f}")

# === Overlay ===
overlay = img_np.copy()
tp = (pred_binary > 0.5) & (mask_np > 0.5)
fp = (pred_binary > 0.5) & (mask_np < 0.5)
fn = (pred_binary < 0.5) & (mask_np > 0.5)
overlay[tp] = [0.0, 1.0, 0.0]
overlay[fp] = [1.0, 0.0, 0.0]
overlay[fn] = [0.0, 0.0, 1.0]

# === Affichage ===
fig, axes = plt.subplots(1, 4, figsize=(22, 6))
fig.suptitle(f"[{SET.upper()}] {sample_file} - Dice: {dice:.4f}", fontsize=12)
axes[0].imshow(img_np);                              axes[0].set_title("Image augmentee");        axes[0].axis("off")
axes[1].imshow(mask_np,   cmap="gray", vmin=0, vmax=1); axes[1].set_title("Points GT (5mm)");        axes[1].axis("off")
axes[2].imshow(pred_np,   cmap="gray", vmin=0, vmax=1); axes[2].set_title("Prediction (sigmoid)");   axes[2].axis("off")
axes[3].imshow(overlay);                             axes[3].set_title("Overlay (V=TP, R=FP, B=FN)"); axes[3].axis("off")
plt.tight_layout(); plt.show(); plt.close()


In [ ]:
# Test du modele sur image reelle (data/output_real/)
import os, glob, cv2
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import segmentation_models_pytorch as smp

%matplotlib inline

# Verification stricte : comparer les poids en memoire vs disk
import glob, os, torch
runs = sorted(glob.glob(os.path.join(cfg.output_dir, "run_*")))
ckpt_path = os.path.join(runs[-1], "checkpoints", "best_model.pth")
ckpt_disk = torch.load(ckpt_path, map_location='cpu', weights_only=False)

# Recuperer un poids specifique en memoire vs disk
weight_name = list(ckpt_disk["model_state_dict"].keys())[5]   # un poids arbitraire
w_mem  = model.state_dict()[weight_name].cpu().flatten()[:5]
w_disk = ckpt_disk["model_state_dict"][weight_name].flatten()[:5]
print(f"Weight tested      : {weight_name}")
print(f"In memory (5 vals) : {w_mem.numpy()}")
print(f"On disk    (5 vals): {w_disk.numpy()}")
print(f"MATCH              : {torch.allclose(w_mem, w_disk)}")
print(f"Ckpt epoch         : {ckpt_disk['epoch']}")
print(f"Ckpt val_dice      : {ckpt_disk.get('val_dice', 'N/A')}")

# === A modifier au besoin ===
REAL_ROOT = r"C:\Users\v\Desktop\ECGPerturb-main\data\output_real"
SUBFOLDER = "photos_crumbles"   # ou None pour scanner tout
INDEX     = 44                        # quel image dans la liste
OVERLAY_COLOR = (255, 0, 0)
OVERLAY_ALPHA = 0.55

# === Liste des images reelles ===
if SUBFOLDER:
    search = os.path.join(REAL_ROOT, SUBFOLDER, "*")
else:
    search = os.path.join(REAL_ROOT, "**", "*")
real_files = sorted([f for f in glob.glob(search, recursive=True)
                     if f.lower().endswith(('.jpg', '.jpeg', '.png', '.webp', '.bmp'))])
print(f"{len(real_files)} images reelles trouvees")

if INDEX >= len(real_files):
    raise IndexError(f"INDEX={INDEX} hors limites (max {len(real_files)-1})")
img_path = real_files[INDEX]
print(f"Image : {img_path}")

# === Chargement du modele (si pas deja en memoire) ===
try:
    model
except NameError:
    runs = sorted(glob.glob(os.path.join(cfg.output_dir, "run_*")))
    if not runs:
        raise FileNotFoundError(f"Aucun run dans {cfg.output_dir}")
    ckpt_path = os.path.join(runs[-1], "checkpoints", "best_model.pth")
    model = smp.Unet(encoder_name=cfg.encoder_name, encoder_weights=None,
                     in_channels=cfg.in_channels, classes=cfg.num_classes)
    ckpt = torch.load(ckpt_path, map_location=cfg.device, weights_only=False)
    model.load_state_dict(ckpt["model_state_dict"])
    model = model.to(cfg.device).eval()
    print(f"Modele charge - epoch {ckpt['epoch']} | val_dice = {ckpt.get('val_dice', 'N/A')}")

# === Preprocessing (identique au training : /255, pas d'imagenet norm) ===
img_pil = Image.open(img_path).convert("RGB")
W_orig, H_orig = img_pil.size
print(f"Resolution native : {W_orig}x{H_orig}")

img_resized = img_pil.resize((cfg.img_width, cfg.img_height), Image.BILINEAR)
img_np = np.array(img_resized, dtype=np.float32) / 255.0
img_tensor = torch.from_numpy(img_np).permute(2, 0, 1).unsqueeze(0).float().to(cfg.device)

# === Inference ===
with torch.no_grad():
    pred_sigmoid = torch.sigmoid(model(img_tensor)).squeeze().cpu().numpy()

pred_full = cv2.resize(pred_sigmoid, (W_orig, H_orig), interpolation=cv2.INTER_LINEAR)
pred_binary = (pred_full > 0.5).astype(np.uint8) * 255

print(f"Heatmap : max={pred_sigmoid.max():.3f}  mean={pred_sigmoid.mean():.4f}")
print(f"Pixels positifs (seuil 0.5) : {(pred_binary>0).sum()} ({100*(pred_binary>0).sum()/pred_binary.size:.2f}%)")

# === Superposition ===
img_arr = np.array(img_pil)
overlay = img_arr.copy().astype(np.float32)
mask_bool = pred_binary > 127
color = np.array(OVERLAY_COLOR, dtype=np.float32)
overlay[mask_bool] = (1 - OVERLAY_ALPHA) * overlay[mask_bool] + OVERLAY_ALPHA * color
overlay = overlay.clip(0, 255).astype(np.uint8)

# === Affichage 4-up ===
fig, axes = plt.subplots(1, 4, figsize=(28, 8))
fig.suptitle(f"Inference reelle : {os.path.basename(img_path)}", fontsize=12)
axes[0].imshow(img_arr);                                                          axes[0].set_title(f"Image reelle ({W_orig}x{H_orig})");           axes[0].axis("off")
axes[1].imshow(pred_full, cmap="hot", vmin=0, vmax=max(pred_full.max(), 1e-6));   axes[1].set_title(f"Sigmoid (max={pred_full.max():.2f})");        axes[1].axis("off")
axes[2].imshow(pred_binary, cmap="gray", vmin=0, vmax=255);                       axes[2].set_title("Mask binarise (seuil 0.5)");                   axes[2].axis("off")
axes[3].imshow(overlay);                                                          axes[3].set_title("Superposition");                               axes[3].axis("off")
plt.tight_layout(); plt.show()